# 03 — Couche gold : les tables pour le persona

silver (nettoyé) → **gold** (agrégé pour le persona) → PostgreSQL → tableau de bord Streamlit

In [1]:
import sys; sys.path.append("/home/jovyan/work")
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from commun.spark_utils import creer_session, format_tables, vers_postgres, LAKE

spark = creer_session("gold")
FMT = format_tables(spark)
spark.conf.set("spark.sql.session.timeZone", "UTC")   # même réglage que dans le silver

conso = spark.read.format(FMT).load(f"{LAKE}/silver/enedis_conso")
meteo = spark.read.format(FMT).load(f"{LAKE}/silver/meteo_regions")
print(f"silver conso : {conso.count():,} lignes")
print(f"silver météo : {meteo.count():,} lignes")

Préparation de Spark 4.2.0 (Hadoop 3.5.0) + Delta Lake + data lake S3 … le premier démarrage peut prendre quelques minutes.
Session prête. Spark UI : http://localhost:4040
silver conso : 2,575,440 lignes
silver météo : 4,380 lignes


## 1. Préparation : la charge totale de chaque région, demi-heure par demi-heure

Le silver a une ligne par profil de client (résidentiel, professionnel…). On additionne les profils pour obtenir la charge totale de la région.

**Piège du changement d'heure :** le 30/10/2022, la tranche 2 h – 3 h existe deux fois en heure de Paris. Pour la puissance, on fait la moyenne de ces doublons (sinon la puissance serait doublée) ; pour l'énergie, on garde la somme.

In [2]:
# Étape A : une ligne par demi-heure, région et profil (les doublons du 30/10 sont fusionnés)
par_profil = (conso
    .groupBy("horodate", "date", "heure", "code_region", "nom_region", "famille_profil", "profil")
    .agg(F.avg("puissance_mw").alias("puissance_mw"),
         F.sum("energie_wh").alias("energie_wh"),
         F.avg("nb_points").alias("nb_points")))

# Étape B : on additionne tous les profils -> charge totale de la région
demi_heure = (par_profil
    .groupBy("horodate", "date", "heure", "code_region", "nom_region")
    .agg(F.sum("puissance_mw").alias("puissance_mw"),
         F.sum("energie_wh").alias("energie_wh"),
         F.sum("nb_points").alias("nb_points")))

demi_heure.cache()
print(f"demi-heures × régions : {demi_heure.count():,}")

demi-heures × régions : 210,216


## 2. gold_conso_jour_region : une ligne par région et par jour, avec la température

In [3]:
jour = (demi_heure
    .groupBy("date", "code_region", "nom_region")
    .agg(F.round(F.sum("energie_wh") / 1e6, 1).alias("energie_mwh"),
         F.round(F.avg("puissance_mw"), 1).alias("puissance_moy_mw"),
         F.round(F.max("puissance_mw"), 1).alias("pointe_mw"),
         F.round(F.avg("nb_points")).cast("long").alias("nb_points")))

gold_jour = (jour
    .join(meteo.select("date", "code_region", "tmoy_c", "tmin_c", "tmax_c"),
          ["date", "code_region"], "inner")
    .withColumn("kwh_par_client", F.round(F.col("energie_mwh") * 1000 / F.col("nb_points"), 2))
    .withColumn("mois", F.month("date"))
    .withColumn("hiver", F.month("date").isin(11, 12, 1, 2, 3))
    .withColumn("week_end", F.dayofweek("date").isin(1, 7))
    .orderBy("code_region", "date"))

gold_jour.write.format(FMT).mode("overwrite").save(f"{LAKE}/gold/conso_jour_region")
g1 = spark.read.format(FMT).load(f"{LAKE}/gold/conso_jour_region")
vers_postgres(g1, "gold_conso_jour_region")
print(f"gold_conso_jour_region : {g1.count():,} lignes (attendu 12 × 365 = 4,380)")
g1.filter(F.col("nom_region") == "Île-de-France").orderBy("tmoy_c").show(5)

Table PostgreSQL « gold_conso_jour_region » écrite.
gold_conso_jour_region : 4,380 lignes (attendu 12 × 365 = 4,380)
+----------+-----------+-------------+-----------+----------------+---------+---------+------+------+------+--------------+----+-----+--------+
|      date|code_region|   nom_region|energie_mwh|puissance_moy_mw|pointe_mw|nb_points|tmoy_c|tmin_c|tmax_c|kwh_par_client|mois|hiver|week_end|
+----------+-----------+-------------+-----------+----------------+---------+---------+------+------+------+--------------+----+-----+--------+
|2022-12-17|         11|Île-de-France|   138089.4|          5753.7|   6796.3|  6583793| -2.09| -3.62| -0.56|         20.97|  12| true|    true|
|2022-12-12|         11|Île-de-France|   138140.9|          5755.9|   7055.9|  6581049| -1.46| -2.88| -0.03|         20.99|  12| true|   false|
|2022-12-11|         11|Île-de-France|   133281.8|          5553.4|   6670.2|  6580391|  -1.4|  -2.9|  0.11|         20.25|  12| true|    true|
|2022-12-13|       

## 3. gold_thermosensibilite_region : de combien la consommation monte quand il fait 1 °C de moins

On garde les jours où la température moyenne est sous 15 °C (au-dessus, on ne chauffe presque plus).
Pour chaque région, on trace la droite qui passe au mieux dans le nuage « température → puissance moyenne ».
Sa pente = le gradient de thermosensibilité : les MW supplémentaires pour chaque degré perdu.

In [4]:
SEUIL_CHAUFFAGE = 15      # °C
T_VAGUE_DE_FROID = -5     # °C : température moyenne d'une vague de froid sévère

froid = g1.filter(F.col("tmoy_c") < SEUIL_CHAUFFAGE)

thermo = (froid
    .groupBy("code_region", "nom_region")
    .agg(F.count("*").alias("nb_jours_sous_15c"),
         (F.covar_pop("tmoy_c", "puissance_moy_mw") / F.var_pop("tmoy_c")).alias("pente_moy"),
         (F.covar_pop("tmoy_c", "pointe_mw") / F.var_pop("tmoy_c")).alias("pente_pointe"),
         F.corr("tmoy_c", "puissance_moy_mw").alias("r"),
         F.avg("tmoy_c").alias("t_moyenne"),
         F.avg("pointe_mw").alias("pointe_moyenne"),
         F.avg("nb_points").alias("nb_clients"))
    .join(g1.groupBy("code_region").agg(F.max("pointe_mw").alias("pointe_2022_mw")), "code_region")
    .withColumn("gradient_mw_par_degre", F.round(-F.col("pente_moy"), 1))
    .withColumn("gradient_w_par_client", F.round(-F.col("pente_moy") * 1e6 / F.col("nb_clients"), 1))
    .withColumn("r2", F.round(F.pow("r", 2), 2))
    .withColumn("pointe_estimee_moins5c_mw",
                F.round(F.col("pointe_moyenne") + F.col("pente_pointe") * (T_VAGUE_DE_FROID - F.col("t_moyenne")), 0))
    .withColumn("depassement_pointe_2022_pct",
                F.round((F.col("pointe_estimee_moins5c_mw") / F.col("pointe_2022_mw") - 1) * 100, 1))
    .withColumn("rang_reactivite",
                F.row_number().over(Window.orderBy(F.desc("gradient_w_par_client"))))
    .select("rang_reactivite", "code_region", "nom_region", "nb_jours_sous_15c",
            "gradient_mw_par_degre", "gradient_w_par_client", "r2",
            "pointe_2022_mw", "pointe_estimee_moins5c_mw", "depassement_pointe_2022_pct",
            F.round("nb_clients").cast("long").alias("nb_clients")))

thermo.write.format(FMT).mode("overwrite").save(f"{LAKE}/gold/thermosensibilite_region")
g2 = spark.read.format(FMT).load(f"{LAKE}/gold/thermosensibilite_region")
vers_postgres(g2, "gold_thermosensibilite_region")
g2.orderBy("rang_reactivite").show(truncate=False)

Table PostgreSQL « gold_thermosensibilite_region » écrite.
+---------------+-----------+--------------------------+-----------------+---------------------+---------------------+----+--------------+-------------------------+---------------------------+----------+
|rang_reactivite|code_region|nom_region                |nb_jours_sous_15c|gradient_mw_par_degre|gradient_w_par_client|r2  |pointe_2022_mw|pointe_estimee_moins5c_mw|depassement_pointe_2022_pct|nb_clients|
+---------------+-----------+--------------------------+-----------------+---------------------+---------------------+----+--------------+-------------------------+---------------------------+----------+
|1              |53         |Bretagne                  |225              |116.6                |53.8                 |0.77|2749.0        |3577.0                   |30.1                       |2166334   |
|2              |28         |Normandie                 |236              |104.0                |52.3                 |0.83|26

## 4. gold_pointe_horaire : à quelle heure tombe la pointe ?

Puissance moyenne par heure de la journée, par région, par famille de clients (RES, PRO, ENT) et par mois.

In [5]:
par_famille = (par_profil
    .groupBy("horodate", "date", "heure", "code_region", "nom_region", "famille_profil")
    .agg(F.sum("puissance_mw").alias("puissance_mw")))

gold_heure = (par_famille
    .withColumn("mois", F.month("date"))
    .groupBy("code_region", "nom_region", "famille_profil", "mois", "heure")
    .agg(F.round(F.avg("puissance_mw"), 1).alias("puissance_moy_mw"))
    .orderBy("code_region", "famille_profil", "mois", "heure"))

gold_heure.write.format(FMT).mode("overwrite").option("overwriteSchema", "true").save(f"{LAKE}/gold/pointe_horaire")
g3 = spark.read.format(FMT).load(f"{LAKE}/gold/pointe_horaire")
vers_postgres(g3, "gold_pointe_horaire")
print(f"gold_pointe_horaire : {g3.count():,} lignes (attendu 12 × 3 × 12 × 24 = 10,368)")
(g3.filter((F.col("nom_region") == "Île-de-France") & (F.col("mois") == 12)
           & (F.col("famille_profil") == "RES"))
   .orderBy("heure").show(24))

Table PostgreSQL « gold_pointe_horaire » écrite.
gold_pointe_horaire : 10,368 lignes (attendu 12 × 3 × 12 × 24 = 10,368)
+-----------+-------------+--------------+----+-----+----------------+
|code_region|   nom_region|famille_profil|mois|heure|puissance_moy_mw|
+-----------+-------------+--------------+----+-----+----------------+
|         11|Île-de-France|           RES|  12|    0|          3607.7|
|         11|Île-de-France|           RES|  12|    1|          3435.7|
|         11|Île-de-France|           RES|  12|    2|          3334.2|
|         11|Île-de-France|           RES|  12|    3|          3044.1|
|         11|Île-de-France|           RES|  12|    4|          2871.7|
|         11|Île-de-France|           RES|  12|    5|          2913.8|
|         11|Île-de-France|           RES|  12|    6|          3178.8|
|         11|Île-de-France|           RES|  12|    7|          3498.4|
|         11|Île-de-France|           RES|  12|    8|          3699.8|
|         11|Île-de-France|

## 5. gold_jours_froids : les 10 jours les plus froids de chaque région

Pour chaque jour, on compare la consommation et la pointe à la moyenne de l'hiver de la région.

In [6]:
rang_froid = Window.partitionBy("code_region").orderBy("tmoy_c")

moy_hiver = (g1.filter(F.col("hiver"))
    .groupBy("code_region")
    .agg(F.avg("puissance_moy_mw").alias("moy_hiver_mw"),
         F.avg("pointe_mw").alias("pointe_moy_hiver_mw")))

gold_froids = (g1
    .withColumn("rang_froid", F.row_number().over(rang_froid))
    .filter(F.col("rang_froid") <= 10)
    .join(moy_hiver, "code_region")
    .withColumn("surplus_conso_pct", F.round((F.col("puissance_moy_mw") / F.col("moy_hiver_mw") - 1) * 100, 1))
    .withColumn("surplus_pointe_pct", F.round((F.col("pointe_mw") / F.col("pointe_moy_hiver_mw") - 1) * 100, 1))
    .select("code_region", "nom_region", "rang_froid", "date", "tmoy_c", "tmin_c",
            "puissance_moy_mw", "pointe_mw", "surplus_conso_pct", "surplus_pointe_pct"))

gold_froids.write.format(FMT).mode("overwrite").save(f"{LAKE}/gold/jours_froids")
g4 = spark.read.format(FMT).load(f"{LAKE}/gold/jours_froids")
vers_postgres(g4, "gold_jours_froids")
print(f"gold_jours_froids : {g4.count():,} lignes (attendu 12 × 10 = 120)")
g4.filter(F.col("nom_region") == "Grand Est").orderBy("rang_froid").show(truncate=False)

Table PostgreSQL « gold_jours_froids » écrite.
gold_jours_froids : 120 lignes (attendu 12 × 10 = 120)
+-----------+----------+----------+----------+------+------+----------------+---------+-----------------+------------------+
|code_region|nom_region|rang_froid|date      |tmoy_c|tmin_c|puissance_moy_mw|pointe_mw|surplus_conso_pct|surplus_pointe_pct|
+-----------+----------+----------+----------+------+------+----------------+---------+-----------------+------------------+
|44         |Grand Est |1         |2022-12-17|-6.2  |-9.98 |2321.4          |2632.0   |34.7             |28.6              |
|44         |Grand Est |2         |2022-12-16|-3.54 |-7.72 |2151.5          |2473.4   |24.9             |20.8              |
|44         |Grand Est |3         |2022-12-12|-3.52 |-5.85 |2292.1          |2685.1   |33.0             |31.2              |
|44         |Grand Est |4         |2022-12-13|-3.46 |-7.0  |2272.7          |2616.2   |31.9             |27.8              |
|44         |Grand Est 

## 6. Bilan du gold

| Table (data lake + PostgreSQL) | Lignes | Contenu |
|---|---|---|
| `gold_conso_jour_region` | 4 380 | Énergie, puissance moyenne, pointe, clients et températures, par région et par jour |
| `gold_thermosensibilite_region` | 12 | Gradient (MW/°C et W/°C par client), r², pointe 2022, pointe estimée à −5 °C, rang |
| `gold_pointe_horaire` | 10 368 | Puissance moyenne par heure, famille de clients et mois |
| `gold_jours_froids` | 120 | Les 10 jours les plus froids par région et leur surplus par rapport à l'hiver moyen |

**Hypothèses et limites :**
- Seuil de chauffage fixé à 15 °C ; relation considérée comme linéaire sous ce seuil.
- La pointe à −5 °C est une extrapolation : peu fiable pour les régions qui n'ont jamais connu ces températures en 2022.
- 2022 est une année atypique (crise de l'énergie, plan de sobriété à partir de l'automne).
- Changement d'heure du 30/10 : puissance moyennée sur les deux tranches 2 h – 3 h.
- Clients ≤ 36 kVA seulement (particuliers et petits professionnels), pas l'industrie.